# Problema 1: Predicción del rendimiento agrícola

Importaciones

In [257]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

Preparación del conjunto de datos

In [258]:
df_ag = pd.read_csv('datos/produccion_agricola.csv')
df_clima = pd.read_csv('datos/condiciones_climaticas_limpios.csv')

In [259]:
#Agrupar clima por país y año
clima_pais = df_clima.groupby(['anio', 'pais']).agg({
    'temperatura_maxima': 'mean',
    'precipitacion_total': 'mean',
    'humedad_relativa_promedio': 'mean',
    'indice_aridez': 'mean',
    'meses_estres_hidrico': 'mean'
}).reset_index()
clima_pais

,anio,pais,temperatura_maxima,precipitacion_total,humedad_relativa_promedio,indice_aridez,meses_estres_hidrico
0,2011,Alemania,18.200000,886.000000,76.00,24.860000,0.000000
1,2011,Argentina,26.133333,795.666667,69.00,14.753333,4.666667
2,2011,Australia,31.360000,975.800000,77.40,14.960000,0.800000
3,2011,Brasil,31.100000,547.000000,61.00,8.372000,8.800000
4,2011,China,25.500000,817.000000,54.00,13.350000,2.000000
...,...,...,...,...,...,...,...
95,2020,Estados Unidos,29.000000,1036.600000,46.20,17.640000,3.800000
96,2020,Francia,21.400000,1387.000000,43.00,27.920000,4.000000
97,2020,India,31.175000,647.000000,57.25,10.035000,5.250000
98,2020,Kenia,35.700000,938.000000,64.00,11.840000,0.000000


In [260]:
# Combinación de datasets
df_master = pd.merge(df_ag, clima_pais, on=['anio', 'pais'], how='inner')
df_master = df_master.dropna()
df_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   pais                       800 non-null    object 
 1   codigo_iso                 800 non-null    object 
 2   region                     800 non-null    object 
 3   cultivo                    800 non-null    object 
 4   anio                       800 non-null    int64  
 5   superficie_hectareas       800 non-null    int64  
 6   rendimiento_ton_ha         800 non-null    float64
 7   produccion_ton             800 non-null    int64  
 8   fertilizantes_kg_ha        800 non-null    int64  
 9   agua_riego_m3_ha           800 non-null    int64  
 10  tendencia_5_anios          800 non-null    float64
 11  temperatura_maxima         800 non-null    float64
 12  precipitacion_total        800 non-null    float64
 13  humedad_relativa_promedio  800 non-null    float64

Division en train y test

In [261]:
# Variable objetivo
y = df_master['rendimiento_ton_ha']

In [262]:
# Vables predictoras
num_features = [
    'fertilizantes_kg_ha', 'agua_riego_m3_ha', 
    'temperatura_maxima', 'precipitacion_total', 'humedad_relativa_promedio', 
    'indice_aridez', 'meses_estres_hidrico'
]
cat_features = ['cultivo']
X = df_master[num_features + cat_features]
X

,fertilizantes_kg_ha,agua_riego_m3_ha,temperatura_maxima,precipitacion_total,humedad_relativa_promedio,indice_aridez,meses_estres_hidrico,cultivo
0,105,4811,26.133333,795.666667,69.0,14.753333,4.666667,Soja
1,164,5557,26.133333,795.666667,69.0,14.753333,4.666667,Maíz
2,105,6514,26.133333,795.666667,69.0,14.753333,4.666667,Cebada
3,236,8527,26.133333,795.666667,69.0,14.753333,4.666667,Té
4,276,6925,26.133333,795.666667,69.0,14.753333,4.666667,Algodón
...,...,...,...,...,...,...,...,...
795,70,6500,35.700000,938.000000,64.0,11.840000,0.000000,Arroz
796,208,1669,35.700000,938.000000,64.0,11.840000,0.000000,Maíz
797,85,6487,35.700000,938.000000,64.0,11.840000,0.000000,Té
798,229,4437,35.700000,938.000000,64.0,11.840000,0.000000,Girasol


In [263]:
# Escalado de las variables para la red de neuronas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
    ])
X_processed = preprocessor.fit_transform(X)

In [264]:
X_processed.shape

(800, 16)

In [265]:
# Division en train y test
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

In [266]:
# Función auxiliar para evaluar modelos
def evaluar_modelo(nombre, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"--- {nombre} ---")
    print(f"MAE:  {mae:.2f} Ton/ha")
    print(f"RMSE: {rmse:.2f} Ton/ha")
    print(f"R2:   {r2:.4f}\n")

- Regresión lineal

In [267]:
# Pipeline para regresión lineal
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('modelo', LinearRegression())
])

In [268]:
# Validación cruzada para regresión lineal
kf = KFold(n_splits=3, shuffle=True, random_state=42)

# Ejecutar Validación Cruzada para obtener R2, MAE y RMSE
cv_r2 = cross_val_score(pipeline_lr, X, y, cv=kf, scoring='r2')
cv_mae = -cross_val_score(pipeline_lr, X, y, cv=kf, scoring='neg_mean_absolute_error')
cv_rmse = np.sqrt(-cross_val_score(pipeline_lr, X, y, cv=kf, scoring='neg_mean_squared_error'))

In [269]:
# RESULTADOS
print("--- Resultados Regresión Lineal Múltiple (Validación Cruzada 5-Folds) ---")
print(f"R2 Promedio:   {cv_r2.mean():.4f} (Desviación: ±{cv_r2.std():.4f})")
print(f"MAE Promedio:  {cv_mae.mean():.2f} Ton/ha")
print(f"RMSE Promedio: {cv_rmse.mean():.2f} Ton/ha")

# 6. EXTRACCIÓN DE COEFICIENTES (Para interpretar el modelo)
pipeline_lr.fit(X, y) # Entrenamos en todos los datos para extraer los coeficientes
modelo = pipeline_lr.named_steps['modelo']
columnas_procesadas = pipeline_lr.named_steps['preprocessor'].get_feature_names_out()

coeficientes = pd.DataFrame({
    'Variable': columnas_procesadas,
    'Coeficiente': modelo.coef_
}).sort_values(by='Coeficiente', key=abs, ascending=False).head(10)

print("\n--- Top Variables Predictoras (Impacto) ---")
print(coeficientes)

--- Resultados Regresión Lineal Múltiple (Validación Cruzada 5-Folds) ---
R2 Promedio:   0.9285 (Desviación: ±0.0118)
MAE Promedio:  3.15 Ton/ha
RMSE Promedio: 7.24 Ton/ha

--- Top Variables Predictoras (Impacto) ---
                       Variable  Coeficiente
9   cat__cultivo_Caña de azúcar    85.499662
12            cat__cultivo_Maíz     3.777350
7            cat__cultivo_Arroz     2.881329
14           cat__cultivo_Trigo     2.403601
10          cat__cultivo_Cebada     1.575461
8             cat__cultivo_Café    -1.337206
5            num__indice_aridez     1.053633
6     num__meses_estres_hidrico     0.812263
2       num__temperatura_maxima     0.760697
15              cat__cultivo_Té    -0.756842


- Red de neuronas

In [270]:
# Construcción de la arquitectura
modelo_ann = Sequential([
    # Capa de entrada y primera capa oculta (64 neuronas, activación ReLU)
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    # Dropout para evitar sobreajuste (apaga el 20% de las neuronas aleatoriamente)
    Dropout(0.2),
    # Segunda capa oculta (32 neuronas)
    Dense(32, activation='relu'),
    Dropout(0.2),
    # Tercera capa oculta (16 neuronas)
    Dense(16, activation='relu'),
    # Capa de salida (1 neurona sin activación, ya que es regresión)
    Dense(1)
])

# Compilación del modelo
modelo_ann.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Configurar parada temprana (Early Stopping) si el modelo deja de mejorar
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# Entrenamiento del modelo
historial = modelo_ann.fit(
    X_train, y_train,
    validation_split=0.2, # Usa 20% del train para validar en cada epoch
    epochs=150,           # Vueltas completas a los datos
    batch_size=32,
    callbacks=[early_stop],
    verbose=1             
)

# Predicción y evaluación
y_pred_ann = modelo_ann.predict(X_test).flatten()
evaluar_modelo("Red Neuronal (ANN)", y_test, y_pred_ann)

Epoch 1/150


c:\Users\Ccp0897\anaconda3\envs\piaentorno\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 928.5666 - mae: 12.9099 - val_loss: 534.6173 - val_mae: 8.1132
Epoch 2/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 910.1260 - mae: 12.1821 - val_loss: 520.4261 - val_mae: 7.4074
Epoch 3/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 880.0035 - mae: 11.4424 - val_loss: 502.1087 - val_mae: 6.9301
Epoch 4/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 842.3077 - mae: 11.3670 - val_loss: 483.0046 - val_mae: 7.5384
Epoch 5/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 799.3867 - mae: 12.2591 - val_loss: 470.8524 - val_mae: 9.0057
Epoch 6/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 763.2756 - mae: 13.6665 - val_loss: 463.9844 - val_mae: 10.7032
Epoch 7/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 737.2032 - mae: 14.9803 - val_loss: 455.1151 - val_mae: 11.6716
Epoch 8/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 682.2288 - mae: 15.1991 - val_loss: 442.3114 - val_mae: 12.2756
Epoch 9/150
16/16 ━━━━━━━━━━━━━━